# FinAI Risk Model — Backtest & v2 Training

Run this notebook after you have at least 30 days of pipeline data.
It will:
1. Load your stored feature vectors from the DB
2. Fetch post-event stock returns (labels) via yfinance
3. Train and evaluate the XGBoost v2 model
4. Save the model — then set `risk.version: v2` in config.yaml


In [ ]:
import sys
sys.path.insert(0, '..')   # run from notebooks/

import yaml
from dotenv import load_dotenv
load_dotenv('../.env')

with open('../config.yaml') as f:
    config = yaml.safe_load(f)

from src.db import init_db
init_db(config['database']['url'].replace('data/', '../data/'))

from src.risk.backtest import (
    load_feature_df, fetch_returns, label_dataset, train_model, garch_baseline
)

event_types = list(config['event_taxonomy'].keys())
tickers     = [c['ticker'] for c in config['companies']]
print(f'Tracking {len(tickers)} tickers | {len(event_types)} event types')

In [ ]:
# Step 1: Load feature vectors from DB
feat_df = load_feature_df(event_types)
print(f'Loaded {len(feat_df)} company-days')
feat_df.head()

In [ ]:
# Step 2: Fetch price returns and label
return_df  = fetch_returns(tickers, feat_df['score_date'].min(), feat_df['score_date'].max())
labeled_df = label_dataset(feat_df, return_df, drop_threshold=-0.02)

import plotly.express as px
fig = px.histogram(labeled_df, x='fwd_return_3d', color='label',
                   title='Distribution of 3-day forward returns (0=no drop, 1=drop >2%)',
                   nbins=50, barmode='overlay', opacity=0.7)
fig.show()

In [ ]:
# Step 3: EDA — do higher v1 scores predict drops?
fig2 = px.box(labeled_df, x='label', y='v1_score',
              title='v1 Risk Score vs actual drop outcome',
              labels={'label': 'Dropped >2% in 3d', 'v1_score': 'v1 Risk Score'})
fig2.show()

In [ ]:
# Step 4: Train v2 model
metrics = train_model(
    labeled_df,
    event_types=event_types,
    output_path='../data/risk_model_v2.joblib',
)

print(f"\nROC-AUC: {metrics['roc_auc']}")
print(f"Avg Precision: {metrics['avg_precision']}")

In [ ]:
# Step 5: Compare against volatility baseline
baseline = garch_baseline(tickers, feat_df)

print('\n=== Model vs Baseline ===')
print(f"v2 XGBoost ROC-AUC:       {metrics['roc_auc']:.4f}")
print(f"Volatility baseline AUC:  {baseline.get('baseline_roc_auc', 'N/A')}")
print("\nPaste these numbers into your README!")

In [ ]:
# Step 6: Feature importance plot
import pandas as pd
feat_imp = pd.DataFrame(metrics['top_features'], columns=['feature', 'importance'])
fig3 = px.bar(feat_imp.sort_values('importance'), x='importance', y='feature',
              orientation='h', title='XGBoost Feature Importances')
fig3.show()
print("\nTo activate v2: set `risk.version: v2` in config.yaml")